# Data preparation

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

![02 / Data Preparation — Application labels and the working inclusion policy](assets/pixel_theme/02_data_preparation.svg)

The audit identified a catalog containing games, software, and media, alongside missing metadata and incomplete app histories. Preparation defines which apps contribute to category activity and how the records are stored, while preserving the observations and reasons behind each decision.

The inputs are the recorded snapshots in data/raw, with versions and checksums documented in data/raw/sources.json.

## 1. Assess application scope

The source has no dedicated app-type field, so labels provide the evidence for a consistent screening rule. The aim is to reduce software and media activity in the game-related cohort, with classification uncertainty carried forward for interpretation.


In [1]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
from notebook_display import configure_notebook, present
configure_notebook()

raw_data = project_root / "data" / "raw"

games = pd.read_parquet(
    raw_data / "fronkon_metadata" / "games.parquet",
    columns=["appID", "name", "genres"],
).rename(columns={"appID": "app_id"})
players = pd.read_csv(
    raw_data / "monthly_players" / "steamcharts.csv",
    usecols=["steam_appid", "name", "month", "avg_players"],
).rename(columns={"steam_appid": "app_id"})
games["app_id"] = pd.to_numeric(games["app_id"], errors="raise").astype("Int64")
players["app_id"] = pd.to_numeric(players["app_id"], errors="raise").astype("Int64")


### Classify application labels

Apps are grouped by game-related and software/media labels, with mixed cases kept separate because broad labels such as Casual or Indie can also appear on software.

The rule uses eight game-related labels alongside a software/media flag list. Indie, Massively Multiplayer, Free To Play, and Early Access describe other characteristics, so apps carrying only those labels remain unresolved under this rule.


In [2]:
game_related_labels = {
    "Action", "Adventure", "Casual", "Simulation", "Strategy", "RPG", "Sports", "Racing",
}
software_media_labels = {
    "Utilities", "Design & Illustration", "Animation & Modeling", "Video Production",
    "Audio Production", "Game Development", "Software Training", "Photo Editing",
    "Web Publishing", "Accounting", "Movie", "Documentary", "Episodic", "Short", "360 Video",
}

has_game_label = games["genres"].map(lambda labels: bool(set(labels) & game_related_labels))
has_software_label = games["genres"].map(lambda labels: bool(set(labels) & software_media_labels))

games["review_group"] = "No decisive label"
games.loc[has_game_label & ~has_software_label, "review_group"] = "Game-related labels only"
games.loc[has_software_label & ~has_game_label, "review_group"] = "Software/media labels only"
games.loc[has_game_label & has_software_label, "review_group"] = "Mixed game/software labels"

tracked_apps = players[["app_id"]].drop_duplicates().merge(
    games, on="app_id", how="left", validate="one_to_one", indicator=True,
)
tracked_apps.loc[tracked_apps["_merge"].eq("left_only"), "review_group"] = "Missing metadata"


### Compare app counts and activity shares

Full-history counts show how many tracked apps fall into each group, while September 2025 activity shares show their contribution to the latest recorded total. Comparing both helps assess the inclusion decision, since a few popular software apps can contribute more activity than many smaller games.


In [3]:
month_dates = pd.to_datetime(players["month"], format="%b-%y", errors="raise")
latest_month = month_dates.max()
latest_players = players.loc[month_dates.eq(latest_month)].merge(
    tracked_apps[["app_id", "review_group"]],
    on="app_id", how="left", validate="one_to_one",
)

scope_summary = tracked_apps.groupby("review_group").agg(tracked_apps=("app_id", "nunique"))
latest_summary = latest_players.groupby("review_group").agg(
    latest_month_apps=("app_id", "nunique"),
    average_concurrent_players=("avg_players", "sum"),
)
latest_summary["activity_share_pct"] = (
    latest_summary["average_concurrent_players"] / latest_players["avg_players"].sum() * 100
)
scope_summary = scope_summary.join(latest_summary)

present(scope_summary.round(2), 'Compare app counts and activity shares')


,Tracked apps,Latest month apps,Summed concurrent activity,Activity share
review_group,,,,
Game-related labels only,"6,094","3,561","3,196,727.72",96.10%
Missing metadata,358,174,"7,491.31",0.23%
Mixed game/software labels,11,9,"80,331.25",2.41%
No decisive label,136,49,"2,953.35",0.09%
Software/media labels only,130,84,"39,005.99",1.17%


### Review flagged apps with the most activity

The following records show the software/media and mixed-label apps contributing the most activity in September 2025. Names help recognize the examples, while their assignments follow the source labels used in screening.


In [4]:
flagged_groups = ["Software/media labels only", "Mixed game/software labels"]
review_examples = latest_players.loc[
    latest_players["review_group"].isin(flagged_groups)
].merge(games[["app_id", "genres"]], on="app_id", how="left", validate="one_to_one")

review_examples = review_examples.sort_values("avg_players", ascending=False)
present(review_examples[["app_id", "name", "genres", "review_group", "avg_players"]].head(10), 'Review flagged apps with the most activity')


## 2. Apply the inclusion policy

Software/media and mixed labels flag **141 tracked apps**, accounting for **3.59%** of observed September 2025 activity. The other unresolved groups contain **136 apps** with inconclusive labels and **358 apps** without matching metadata.

The working cohort includes apps with at least one game-related label and no software/media flag, selecting **6,094 of 6,729 tracked apps** and approximately **96.10%** of latest-month recorded activity.

The **635 held-out apps** remain stored with an eligibility flag and reason, allowing the rule to be reviewed against the same evidence. Genuine games with insufficient or mixed labels may be excluded, which remains a limitation of this conservative label-based cohort.


## 3. Structure the analytical data

App attributes, label assignments, and monthly observations are stored separately, with each table's grain defining what one row represents and which columns identify it. The relationships preserve those meanings when records are joined.

| Table | Grain | Primary key | Reference |
|---|---|---|---|
| games | One tracked app | app_id | metadata_source_id references source_runs |
| game_genres | One app/label assignment | app_id, genre | app_id references games |
| monthly_players | One observed app/month | app_id, month | app_id references games; source_run_id references source_runs |
| source_runs | One recorded input snapshot per source in this build | source_run_id | Source URL, version/revision, and manifest entry |
| analysis_categories | One selected analysis label | genre | Matched to game_genres.genre during aggregation |

### Relationships

```text
source_runs
    |-- metadata_source_id --> games
    |                            |-- app_id --> game_genres
    |                            |-- app_id --> monthly_players
    |-- source_run_id -----------------------> monthly_players

analysis_categories.genre -- matches --> game_genres.genre
```

An app can have several monthly observations and several labels, with game_genres connecting apps to categories through distinct app/label pairs. Composite keys protect observations and assignments from duplication at their respective grains.

The history-to-label join attributes full activity to each applicable category, so category totals overlap and need to be interpreted individually.

## 4. Preserve the meaning of observations

Preparation standardizes IDs as positive integers, dates as month starts, and labels as unique trimmed values, while rejecting invalid activity and duplicate keys. Recorded zeros remain zero and missing app/month records remain absent from the observation table.

The category query builds a continuous calendar, leaving activity unknown when no category apps are observed. Its coverage measure compares observed apps with eligible apps whose first-to-last recorded spans include that month, describing internal completeness and leaving periods outside those spans unmeasured.

At the latest month, 100% coverage follows from the span definition, while apps that disappeared earlier are outside the denominator. Sample stability and representation of Steam therefore need separate consideration.

## 5. Preparation outcome

The prepared snapshot retains **6,729 apps and 612,265 monthly observations**, with five overlapping categories producing **795 category/month rows** across July 2012–September 2025.

The pipeline verifies source checksums and database integrity, while [database validation](03_database_validation.ipynb) presents the executed build, scope-impact results, and SQL/pandas comparison.

These checks support exploration of activity trajectories, changes in the observed sample, and gaps, with newer history needed for an up-to-date 2027 forecast.
